# Fusion-strategy benchmark — baseline vs S1–S4

Head-to-head comparison of the 4 anti-modality-collapse strategies against the current default, mirroring `sweep_benchmark.ipynb` (UMAPs, PPC, Moran's I, scib, ranking) and adding the **spatial latent-usage** headline metric.

**Trains all 5 models inline — run on a GPU kernel.** Same data/keys as the sweep, so results are comparable.

| strategy | mechanism |
|---|---|
| baseline | current default (POE) |
| S1 | MoE weight-floor + entropy regulariser |
| S2 | PoE precision temperature |
| S3 | MVTCAE total-correlation objective |
| S4 | MMVAE+ shared/private + cross-reconstruction |

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append('/home/projects/nyosef/zvise/PixelGen/')

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from matplotlib import pyplot as plt

from PixelGen.multimodalvi import MultiModalSCVI
from PixelGen.enums import D
from PixelGen.pxl_utils import train_model
from PixelGen.scvi_utils import plot_losses, pca_neighbors_umap
from PixelGen.utils import plot_composite_ppc, calculate_metrics, get_dense
from PixelGen.metrics import distr_autocorrelation_in_latent, spatial_latent_usage

sc.set_figure_params(figsize=(5, 3), frameon=False)

ROOT       = Path('/home/projects/nyosef/zvise/PixelGen/PixelGen')
CACHE      = ROOT / 'New_Data' / 'cache'
ADATA_PATH = CACHE / 'adata_cytovi_annotated_compat.h5ad'

ABUNDANCE_LAYER = 'arcsinh'
SPATIAL_KEY     = 'spatial_asinh5_top500var'
BATCH_KEY       = 'cell_system'
BIO_KEY         = 'cell_type_annot'

## Config

Shared base config (from `sweep/train_sweep.py`); each strategy overrides via `STRATEGIES`. Defaults reproduce current behaviour, so `baseline` = today's model.

In [ ]:
setup_kwargs = dict(layer=ABUNDANCE_LAYER, extra_modality_keys=[SPATIAL_KEY],
                    n_modalities=2, batch_key=BATCH_KEY, spatial_mask_key=None)

base_model_kwargs = dict(n_latent=20, n_hidden=128, n_layers=2, dropout_rate=0.1,
                         distrs=[D.Normal, D.Normal], loss_weights='auto',
                         joint_kl=False, unimodal_kl=True, batch_mask=[False, True])

train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True,
                    early_stopping_patience=200, batch_size=2000, max_epochs=10000,
                    plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400))

STRATEGIES = {
    'baseline':           dict(experts_method='POE'),
    'S1_floor_entropy':   dict(experts_method='MOE', weight_floor=0.2, weight_entropy_reg=0.1),
    'S2_poe_temperature': dict(experts_method='POE', poe_calibration='temperature'),
    'S3_mvtcae':          dict(experts_method='POE', objective='mvtcae', tc_alpha=0.5, tc_beta=1.0),
    'S4_mmvaeplus':       dict(experts_method='POE', n_private=[10, 10],
                               cross_reconstruction=True, cross_recon_weight=1.0, free_bits=0.5),
}
names = list(STRATEGIES)
latent_keys = [f'z__{n}' for n in names]

In [ ]:
adata = sc.read_h5ad(ADATA_PATH)
assert SPATIAL_KEY in adata.obsm, f'{SPATIAL_KEY} missing from obsm'
adata

## Train all strategies (GPU)
Each uses the same setup/train kwargs; only `model_kwargs` differ. Joint latent stored in `obsm['z__<name>']`.

In [ ]:
MODELS_DIR    = CACHE / 'fusion_models'
FORCE_RETRAIN = False   # True -> ignore cache, retrain + overwrite all
MODELS_DIR.mkdir(parents=True, exist_ok=True)

models = {}
for name, extra in STRATEGIES.items():
    path = MODELS_DIR / name
    if path.exists() and not FORCE_RETRAIN:
        print(f'=== loading cached {name} from {path} ===')
        model = MultiModalSCVI.load(str(path), adata=adata)
    else:
        print(f'=== training {name}: {extra} ===')
        model = train_model(adata, MultiModalSCVI, setup_kwargs,
                            {**base_model_kwargs, **extra}, train_kwargs,
                            generate_loss_plots=False)
        model.save(str(path), overwrite=True)
    models[name] = model
    adata.obsm[f'z__{name}'] = model.get_latent_representation(adata, modality='joint')

## 1. Loss curves
(S1 also logs `weight_entropy_penalty`; S4 reconstruction includes the cross-recon terms.)

In [ ]:
for name, model in models.items():
    plot_losses(model)
    plt.suptitle(name, y=1.02)
    plt.show()

## 2. UMAPs of the joint latent
Coloured by cell type (bio) and cell system (batch).

In [ ]:
for name in names:
    pca_neighbors_umap(
        adata, latent_name=f'z__{name}',
        umap_pl_kwargs=dict(color=[BIO_KEY, BATCH_KEY], frameon=False, ncols=2),
        umap_title=name,
    )
    plt.show()

## 3. PPC — per-modality self-reconstruction

In [ ]:
rows = []
for name, model in models.items():
    out = model.get_normalized_expression(
        adata=adata, return_mean_expression=True,
        return_l2_error=False, return_px_distrs=False, return_numpy=True,
    )
    rows += calculate_metrics(get_dense(adata.layers[ABUNDANCE_LAYER]),
                              get_dense(out['exprs'][ABUNDANCE_LAYER]), name, 'Abundance')
    rows += calculate_metrics(get_dense(adata.obsm[SPATIAL_KEY]),
                              get_dense(out['exprs'][SPATIAL_KEY]), name, 'Spatial')
metrics_df = pd.DataFrame(rows)
plot_composite_ppc('Abundance', metrics_df, 'cornflowerblue', names); plt.show()
plot_composite_ppc('Spatial',   metrics_df, 'lightgreen',     names); plt.show()

## 4. Moran's I autocorrelation
kNN graph on each joint latent, Moran's I on abundance + spatial features. Higher spatial = the latent organizes cells along spatial axes. The **abundance** entry is a reference: spatial Moran's I on a graph built from abundance alone — fusion latents above it genuinely add spatial structure beyond what abundance already captures.

In [ ]:
# Reference: abundance-only embedding. Spatial Moran's I on a kNN graph built from abundance alone
# is the baseline each fusion latent must beat to count as genuinely incorporating spatial structure.
adata.obsm['abundance'] = get_dense(adata.layers[ABUNDANCE_LAYER])
spatial_latent_keys = latent_keys + ['abundance']
spatial_names       = names + ['abundance']

autocorr_abundance = distr_autocorrelation_in_latent(
    adata, latent_keys=latent_keys, names=names, rep_key=ABUNDANCE_LAYER, pca_kwargs={'n_comps': 15})
autocorr_spatial = distr_autocorrelation_in_latent(
    adata, latent_keys=spatial_latent_keys, names=spatial_names, rep_key=SPATIAL_KEY, pca_kwargs={'n_comps': 15})
print('abundance:', autocorr_abundance.shape, '| spatial:', autocorr_spatial.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(data=autocorr_spatial, x='morans', hue='latent', kde=True, stat='density',
             common_norm=False, alpha=0.4, ax=axes[0])
axes[0].set_title(f"Spatial Moran's I  ({SPATIAL_KEY} features over each embedding; abundance = ref)")
axes[0].set_xlabel("Moran's I")
sns.histplot(data=autocorr_abundance, x='morans', hue='latent', kde=True, stat='density',
             common_norm=False, alpha=0.4, ax=axes[1], legend=False)
axes[1].set_title(f" Moran's I over ({ABUNDANCE_LAYER} ")
axes[1].set_xlabel("Moran's I")
plt.tight_layout(); plt.show()

In [ ]:
morans_mean = autocorr_spatial.groupby('latent')['morans'].mean().reindex(spatial_names)
morans_mean.sort_values(ascending=False)

## 5. scib metrics
bio = `cell_type_annot`, batch = `cell_system`.

In [ ]:
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection
bm = Benchmarker(
    adata, batch_key=BATCH_KEY, label_key=BIO_KEY, embedding_obsm_keys=latent_keys,
    bio_conservation_metrics=BioConservation(), batch_correction_metrics=BatchCorrection(),
)
bm.benchmark()
bm.plot_results_table(min_max_scale=False)

In [ ]:
scib_df = bm.get_results(min_max_scale=False)
scib_df

## 6. **Headline** — spatial latent-usage ablation
Replace the spatial input with the missing sentinel and measure the model's response. A model that truly fuses spatial degrades on all three (ΔELBO↑, cosine shift↑, Moran's drop↑).

In [ ]:
usage_rows = []
for name, model in models.items():
    r = spatial_latent_usage(model, adata, spatial_key=SPATIAL_KEY, sentinel='missing').iloc[0]
    r['strategy'] = name
    usage_rows.append(r)
usage_df = pd.DataFrame(usage_rows).set_index('strategy').reindex(names)
usage_df.round(4)

## Summary & ranking
rows = {baseline, S1–S4}; combined rank by spatial Moran's I + scib Total (as in `sweep_benchmark`).

In [ ]:
res = scib_df.drop('Metric Type', errors='ignore')
res.index = res.index.str.replace('^z__', '', regex=True)

summary = pd.DataFrame(index=names)
summary['spatial_morans_I'] = morans_mean
if 'Bio conservation' in res.columns:
    summary['scib_bio']    = res['Bio conservation'].astype(float).reindex(names)
    summary['scib_batch']  = res['Batch correction'].astype(float).reindex(names)
    summary['scib_total']  = res['Total'].astype(float).reindex(names)
summary['delta_elbo_ablation']   = usage_df['delta_elbo']
summary['latent_shift_ablation'] = usage_df['latent_cosine_shift']
summary['morans_drop_ablation']  = usage_df['morans_drop']

summary['morans_rank'] = summary['spatial_morans_I'].rank(ascending=False)
if 'scib_total' in summary:
    summary['scib_rank'] = summary['scib_total'].rank(ascending=False)
    summary['mean_rank'] = summary[['morans_rank', 'scib_rank']].mean(axis=1)
    summary = summary.sort_values('mean_rank')
summary.round(4)

## S4 — spatial-private block ("what abundance misses")
The MMVAE+ spatial-private latent should capture spatially-defined structure abundance can't explain. Cluster/colour it to look for states the joint embedding alone hides.

In [ ]:
m4 = models['S4_mmvaeplus']
adata.obsm['z_S4_spatial_private'] = m4.get_latent_representation(adata, representation='private', modality=SPATIAL_KEY)
adata.obsm['z_S4_joint_private']   = m4.get_latent_representation(adata, representation='joint_private')
pca_neighbors_umap(
    adata, latent_name='z_S4_spatial_private',
    umap_pl_kwargs=dict(color=[BIO_KEY], frameon=False),
    umap_title='S4 spatial-private block',
)
plt.show()